In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from transformer_lens import HookedTransformer
import torch
import numpy as np

import json
import time
import os

from src.neuron_heads import activation_steering_based_head_attribution
from src.datahandlers import ActivatingDataset
from src.utils import tuple_str_to_tuple

In [3]:
filename = "2026-06-11_19-07-52_gpt2-small_train"
is_random = "" # otherwise, ""
train = True


with open(f'../experiment_data/3_trimmed_texts/{filename}.json') as f:
    text_metadata = json.load(f)

neurons = [tuple_str_to_tuple(x) for x in text_metadata['neuron_to_trunc_data'].keys()]

In [4]:
import pickle
dataset_text_test_path = "../experiment_data/text_dict_test.pkl"
dataset_text_train_path = "../experiment_data/text_dict_train.pkl"
if train:
    with open(dataset_text_train_path, 'rb') as f:
        dataset = pickle.load(f)
    print("Train loaded!")

else:
    with open(dataset_text_test_path, 'rb') as f:
        dataset = pickle.load(f)
    # removing samples that were also in the training dataset
    dataset.pop(9272) 
    dataset.pop(7138)
    print("Test loaded!")
    

data = ActivatingDataset(text_metadata['neuron_to_trunc_data'], dataset)
data.remove_prompts_longer_than(100)

Train loaded!


In [5]:
# Parameters
parameters = text_metadata['parameters']
model_name = parameters['model_name']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
# Extracting the total number of active heads obtained by the dot-product-based head attribution
def extract_total_active_heads(filename):
    with open(f'../experiment_data/4_head_attributions/{filename}.json') as f:
        neuron_prompt_head_attribution = json.load(f)
    head_attributions = neuron_prompt_head_attribution['head_attributions']
    total_active_heads = 0
    for neuron in head_attributions:
        total_active_heads += sum([len(head_attributions[neuron][prompt]) for prompt in head_attributions[neuron]])
    return total_active_heads  

filename_old = "2026-06-15_03-25-48_gpt2-small_train"
total_active_heads = extract_total_active_heads(filename_old)

In [7]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [8]:
model = HookedTransformer.from_pretrained(
    model_name,
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    # refactor_factored_attn_matrices=True,
    device=device,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [9]:
with torch.no_grad():
    head_attribution_dict, neuron_prompt_head_scores, last_seq_neuron_activations, last_seq_neuron_activations_boosted_head, remaining_heads = activation_steering_based_head_attribution(
        model=model,
        data=data,
        device=device,
        neurons=neurons,
        total_active_heads=int(total_active_heads*1.1),
        boost_factor=2,
        batch_size=16
    )

  0%|          | 0/60 [00:00<?, ?it/s]

  2%|▏         | 1/60 [00:10<10:09, 10.33s/it]

--> statistics of (9, 5) delta activations for prompt 0: mean: -0.0858212855954965/ std: 0.517617913053241/ positive_deltas: 43/ negative/null_deltas: 77
activations of neuron (9, 5) for prompt_idx 0: [0.5686626434326172, 0.29816579818725586, 0.23085260391235352, 0.21336841583251953, 0.20978498458862305, 0.20798206329345703, 0.1972208023071289, 0.16946649551391602, 0.1493840217590332, 0.1483311653137207, 0.1360611915588379, 0.11565876007080078, 0.11023855209350586, 0.09508228302001953, 0.08230972290039062, 0.07950353622436523, 0.07546758651733398, 0.0722346305847168, 0.06418323516845703, 0.05510568618774414, 0.053925514221191406, 0.04338359832763672, 0.03727436065673828, 0.03351402282714844, 0.03299760818481445, 0.0315089225769043, 0.02970409393310547, 0.02718210220336914, 0.023920536041259766, 0.021988868713378906, 0.01945018768310547, 0.01710367202758789, 0.01581430435180664, 0.01479959487915039, 0.014476299285888672, 0.014375686645507812, 0.013111591339111328, 0.010623931884765625, 

  3%|▎         | 2/60 [00:20<09:42, 10.05s/it]

--> statistics of (9, 1616) delta activations for prompt 0: mean: -0.08009075323740641/ std: 0.46979917568834717/ positive_deltas: 40/ negative/null_deltas: 80
activations of neuron (9, 1616) for prompt_idx 0: [0.4187808036804199, 0.17952394485473633, 0.14026451110839844, 0.13553142547607422, 0.12327957153320312, 0.12070178985595703, 0.10885190963745117, 0.10610437393188477, 0.08932065963745117, 0.08713769912719727, 0.08659124374389648, 0.08064985275268555, 0.06682348251342773, 0.057355403900146484, 0.0564723014831543, 0.05535554885864258, 0.053038597106933594, 0.05126142501831055, 0.04642295837402344, 0.04471921920776367, 0.04120492935180664, 0.03661775588989258, 0.035901546478271484, 0.03130149841308594, 0.03011608123779297, 0.029163360595703125, 0.026235103607177734, 0.02516794204711914, 0.024988651275634766, 0.019006729125976562, 0.017993927001953125, 0.015320301055908203, 0.011846065521240234, 0.010679244995117188, 0.010117053985595703, 0.009880542755126953, 0.008677959442138672, 

...

--> statistics of (11, 907) delta activations for prompt 0: mean: -0.026076214181052312/ std: 0.14484276649882125/ positive_deltas: 62/ negative/null_deltas: 82
activations of neuron (11, 907) for prompt_idx 0: [0.24509000778198242, 0.1820220947265625, 0.168471097946167, 0.11245489120483398, 0.1027836799621582, 0.09023857116699219, 0.07212543487548828, 0.06934046745300293, 0.06496191024780273, 0.06340718269348145, 0.06340169906616211, 0.06272363662719727, 0.06129932403564453, 0.05681967735290527, 0.056357383728027344, 0.055251121520996094, 0.051792144775390625, 0.047832489013671875, 0.04325675964355469, 0.03825116157531738, 0.037970781326293945, 0.03777170181274414, 0.03722882270812988, 0.03511524200439453, 0.0350341796875, 0.032971858978271484, 0.030998945236206055, 0.02822732925415039, 0.027965307235717773, 0.02730393409729004, 0.025638103485107422, 0.025190114974975586, 0.022493839263916016, 0.02220892906188965, 0.019046783447265625, 0.01860499382019043, 0.018294334411621094, 0.0167

 98%|█████████▊| 59/60 [10:14<00:10, 10.84s/it]

--> statistics of (11, 2763) delta activations for prompt 0: mean: -0.028421276145511203/ std: 0.44453883135497657/ positive_deltas: 60/ negative/null_deltas: 84
activations of neuron (11, 2763) for prompt_idx 0: [1.3033151626586914, 1.2698163986206055, 1.1325864791870117, 1.0232105255126953, 1.0149612426757812, 0.9864559173583984, 0.8280549049377441, 0.7344851493835449, 0.7280788421630859, 0.6847100257873535, 0.6791629791259766, 0.6267805099487305, 0.5872979164123535, 0.5063800811767578, 0.4636497497558594, 0.45618295669555664, 0.4451603889465332, 0.4445514678955078, 0.4153766632080078, 0.39867258071899414, 0.30919694900512695, 0.2970247268676758, 0.28261470794677734, 0.2674570083618164, 0.2047872543334961, 0.19201946258544922, 0.1845841407775879, 0.1673412322998047, 0.1523571014404297, 0.142425537109375, 0.10555171966552734, 0.09882068634033203, 0.08747386932373047, 0.08511877059936523, 0.08386898040771484, 0.08217334747314453, 0.08092832565307617, 0.08040904998779297, 0.078454971313

100%|██████████| 60/60 [10:34<00:00, 10.57s/it]

--> statistics of (11, 2546) delta activations for prompt 0: mean: -0.04410362243652344/ std: 0.1742970625399362/ positive_deltas: 63/ negative/null_deltas: 81
activations of neuron (11, 2546) for prompt_idx 0: [0.39855241775512695, 0.27285218238830566, 0.16748046875, 0.13849139213562012, 0.1258089542388916, 0.0994880199432373, 0.0693359375, 0.06907463073730469, 0.06688809394836426, 0.05527186393737793, 0.054480791091918945, 0.047086477279663086, 0.03978395462036133, 0.03928494453430176, 0.03894639015197754, 0.03868460655212402, 0.03526163101196289, 0.0340571403503418, 0.03078746795654297, 0.029445886611938477, 0.02778458595275879, 0.027771711349487305, 0.027498722076416016, 0.026459693908691406, 0.025673866271972656, 0.022139549255371094, 0.021611928939819336, 0.021238327026367188, 0.020273923873901367, 0.018413782119750977, 0.018227577209472656, 0.017120361328125, 0.016376972198486328, 0.015629053115844727, 0.014966964721679688, 0.01459646224975586, 0.014481306076049805, 0.0137588977

In [10]:
remaining_heads

0

In [11]:
output = {
    'parameters': parameters,
    'head_attributions': head_attribution_dict,
    'prior_filename': filename,
}

train_str = "train" if train else "test"

timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime(int(time.time())))
new_filename = f"{timestamp}_{model_name}_{train_str}.json"

folder_path = "../experiment_data/4_new_head_attributions"
os.makedirs(folder_path, exist_ok=True)

with open(os.path.join(folder_path, new_filename) , 'w') as f:
    json.dump(output, f)

In [12]:
output = {
    'parameters': parameters,
    'neuron_prompt_head_scores': neuron_prompt_head_scores,
    'prior_filename' : new_filename
}

train_str = "train" if train else "test"

timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime(int(time.time())))
new_filename = f"{timestamp}_{model_name}_{train_str}_activation_steering.json"

folder_path = "../experiment_data/5_head_attribution_scores"
os.makedirs(folder_path, exist_ok=True)

with open(os.path.join(folder_path, new_filename) , 'w') as f:
    json.dump(output, f)